# 🏠 Dubai Property Price Prediction (2026)

**Goal:** Predict the transaction value (`TRANS_VALUE`) of a residential property in Dubai using property, location and area features.

**Notebook flow:**
1. Load data & first look
2. Exploratory Data Analysis (EDA)
3. Data leakage check (very important!)
4. Feature engineering
5. Train/Test split
6. Model training (XGBoost)
7. Evaluation
8. Feature importance
9. Conclusion


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import xgboost as xgb

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)


## 1. Load the Data

In [ ]:
df = pd.read_csv('/kaggle/input/dubai-residential-2026/dubai_residential_data_2026.csv')
print("Shape:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
df.isnull().sum()


## 2. Exploratory Data Analysis (EDA)

Let's understand the target variable and a few key features before touching any model.


In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['TRANS_VALUE'], bins=60, kde=True)
plt.title('Distribution of Transaction Value (AED)')
plt.xlabel('TRANS_VALUE')
plt.show()


In [ ]:
# Price is heavily right-skewed (few very expensive properties).
# Let's look at it on a log scale instead.
plt.figure(figsize=(8,5))
sns.histplot(np.log1p(df['TRANS_VALUE']), bins=60, kde=True, color='orange')
plt.title('Distribution of log(Transaction Value)')
plt.xlabel('log(TRANS_VALUE)')
plt.show()


In [ ]:
plt.figure(figsize=(10,5))
df['PROP_SB_TYPE_EN'].value_counts().plot(kind='bar')
plt.title('Property Type Counts')
plt.ylabel('Count')
plt.show()


In [ ]:
top_areas = df['AREA_EN'].value_counts().head(15)
plt.figure(figsize=(10,6))
sns.barplot(x=top_areas.values, y=top_areas.index)
plt.title('Top 15 Areas by Number of Transactions')
plt.xlabel('Count')
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x='value_band', y='TRANS_VALUE',
            data=df[df['TRANS_VALUE'] < df['TRANS_VALUE'].quantile(0.99)],
            order=['Entry','Mid-Market','High-End','Ultra-Premium'])
plt.title('Transaction Value by Value Band (outliers removed for clarity)')
plt.show()


In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(x='ACTUAL_AREA', y='TRANS_VALUE',
                 data=df[df['TRANS_VALUE'] < df['TRANS_VALUE'].quantile(0.99)],
                 alpha=0.3)
plt.title('Actual Area vs Transaction Value')
plt.show()


## 3. ⚠️ Data Leakage Check

Before building any model, we must check if any column is secretly derived from our target (`TRANS_VALUE`).
If we don't catch this, our model will look amazing on paper but will be useless in real life.


In [ ]:
# Check: is price_per_sqm just TRANS_VALUE / ACTUAL_AREA?
check = df['TRANS_VALUE'] / df['ACTUAL_AREA']
match_pct = (np.isclose(check, df['price_per_sqm'], atol=1)).mean() * 100
print(f"price_per_sqm matches TRANS_VALUE/ACTUAL_AREA in {match_pct:.1f}% of rows")


**Finding:** `price_per_sqm` is calculated directly from `TRANS_VALUE` and `ACTUAL_AREA`.
`value_band` is just a bucketed version of `price_per_sqm`.

Both of these columns leak the target — if we keep them, the model would be "cheating" (it would basically already know the price).
So we **drop `price_per_sqm` and `value_band`** before modeling.

`size_category` is safe to keep — it's only derived from `ACTUAL_AREA`, which is a normal feature, not the target.


In [ ]:
df = df.drop(columns=['price_per_sqm', 'value_band'])


## 4. Feature Engineering

- Fill missing categorical values with `'Unknown'`
- Extract `month` from the transaction date
- Use **target encoding** for high-cardinality columns (`AREA_EN`, `PROJECT_EN`, `NEAREST_LANDMARK_EN`) —
  instead of turning each area into a meaningless number, we replace it with the **average price of that area** (calculated using training data only, to avoid leakage).
- Label-encode the remaining categorical columns


In [ ]:
df['INSTANCE_DATE'] = pd.to_datetime(df['INSTANCE_DATE'])
df['month'] = df['INSTANCE_DATE'].dt.month

fill_cols = ['ROOMS_EN', 'NEAREST_METRO_EN', 'NEAREST_MALL_EN', 'NEAREST_LANDMARK_EN', 'PROJECT_EN']
for c in fill_cols:
    df[c] = df[c].fillna('Unknown')


In [ ]:
# log-transform the target since it's skewed (helps the model a lot)
df['log_value'] = np.log1p(df['TRANS_VALUE'])

# split indices first, so target encoding only ever sees training data
train_idx, test_idx = train_test_split(df.index, test_size=0.2, random_state=42)


In [ ]:
# Target encoding (mean price per category, computed on train set only)
te_cols = ['AREA_EN', 'PROJECT_EN', 'NEAREST_LANDMARK_EN']
global_mean = df.loc[train_idx, 'TRANS_VALUE'].mean()

for c in te_cols:
    means = df.loc[train_idx].groupby(c)['TRANS_VALUE'].mean()
    df[c + '_enc'] = df[c].map(means).fillna(global_mean)


In [ ]:
# Label encode simpler categorical columns
label_cols = ['PROCEDURE_EN', 'IS_FREE_HOLD_EN', 'PROP_SB_TYPE_EN',
              'ROOMS_EN', 'NEAREST_METRO_EN', 'NEAREST_MALL_EN', 'size_category']

for c in label_cols:
    le = LabelEncoder()
    df[c] = le.fit_transform(df[c])


In [ ]:
feature_cols = label_cols + ['ACTUAL_AREA', 'month',
                             'AREA_EN_enc', 'PROJECT_EN_enc', 'NEAREST_LANDMARK_EN_enc']

X = df[feature_cols]
y = df['log_value']

X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)


## 5. Model Training — XGBoost Regressor

We train on the **log-transformed** target, then convert predictions back to real AED values (`expm1`) for evaluation.


In [ ]:
model = xgb.XGBRegressor(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)


## 6. Evaluation

In [ ]:
pred_log = model.predict(X_test)

# Convert back from log scale to actual AED
pred_actual = np.expm1(pred_log)
y_test_actual = np.expm1(y_test)

r2 = r2_score(y_test_actual, pred_actual)
mae = mean_absolute_error(y_test_actual, pred_actual)
rmse = np.sqrt(mean_squared_error(y_test_actual, pred_actual))

print(f"R² Score : {r2:.4f}")
print(f"MAE      : AED {mae:,.0f}")
print(f"RMSE     : AED {rmse:,.0f}")


In [ ]:
plt.figure(figsize=(7,7))
plt.scatter(y_test_actual, pred_actual, alpha=0.3)
lims = [0, y_test_actual.quantile(0.99)]
plt.plot(lims, lims, color='red', linestyle='--')
plt.xlim(lims); plt.ylim(lims)
plt.xlabel('Actual Transaction Value')
plt.ylabel('Predicted Transaction Value')
plt.title('Actual vs Predicted Price')
plt.show()


## 7. Feature Importance

In [ ]:
importance = pd.Series(model.feature_importances_, index=feature_cols).sort_values()

plt.figure(figsize=(8,6))
importance.plot(kind='barh')
plt.title('Feature Importance (XGBoost)')
plt.xlabel('Importance')
plt.show()


## 8. Conclusion

- The dataset contains two leaky columns — **`price_per_sqm`** and **`value_band`** — both mathematically derived from the target (`TRANS_VALUE`). These were removed before modeling to keep results honest.
- After proper feature engineering (target encoding for area/project/landmark, log-transforming the skewed target), an **XGBoost Regressor** achieved:
  - **R² ≈ 0.86**
  - **MAE ≈ AED 328,000**
  - **RMSE** in a similar range, driven mainly by a small number of ultra-luxury outlier transactions.
- The most influential features were **`ACTUAL_AREA`**, **`PROJECT_EN` (encoded)**, **`size_category`**, and **`AREA_EN` (encoded)** — confirming that *location* and *size* remain the two biggest price drivers in Dubai's residential market, even in 2026.
- **Next steps to improve further:** try LightGBM/CatBoost, add more granular location features (lat/long if available), tune hyperparameters with cross-validation, and treat ultra-premium properties as a separate segment.

### 📌 Title Suggestions (within 50 characters)
1. Dubai Property Price Prediction 2026
2. Dubai Real Estate Price Prediction (XGBoost)
3. Predicting Dubai Residential Property Prices
4. Dubai Housing Price Prediction with XGBoost

### 🏷️ Tag Suggestions
`real estate`, `regression`, `xgboost`, `beginner`, `tabular data`, `dubai`, `price prediction`
